In [1]:
import os
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

Based on correlation analysis, all 13 features are retained for tree-based models
(Random Forest, XGBoost) as they perform internal feature selection via split
importance. For Logistic Regression, all features are retained given the small
feature count, with feature importance revisited post-training via SHAP (Step 8).

In [2]:
df = pd.read_csv(os.path.join('..', 'data', 'processed', 'heart_disease.csv'))
df.shape

(908, 14)

In [3]:
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_cols = ['cp', 'restecg', 'slope', 'thal', 'ca']


encoder = OneHotEncoder(sparse_output=False)
scaler = StandardScaler()

SPLITS_DIR = os.path.join('..', 'data', 'processed', 'splits')
shutil.rmtree(SPLITS_DIR, ignore_errors=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

In [4]:
X = df.drop(columns=['target'])
y = df['target']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

In [6]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

target
1    0.546832
0    0.453168
Name: proportion, dtype: float64
target
1    0.543956
0    0.456044
Name: proportion, dtype: float64


In [7]:
X_train_lr = encoder.fit_transform(X_train[categorical_cols])
X_test_lr = encoder.transform(X_test[categorical_cols])

X_train_lr = pd.DataFrame(X_train_lr, columns=encoder.get_feature_names_out(categorical_cols))
X_test_lr = pd.DataFrame(X_test_lr, columns=encoder.get_feature_names_out(categorical_cols))

X_train_lr = pd.concat([X_train.drop(columns=categorical_cols), X_train_lr], axis=1)
X_test_lr = pd.concat([X_test.drop(columns=categorical_cols), X_test_lr], axis=1)

print(X_train_lr)
print(X_test_lr)

      age  sex  trestbps   chol  fbs  thalach  exang  oldpeak  cp_1.0  cp_2.0  \
0    74.0  1.0     140.0  237.0  1.0     94.0    0.0      0.0     0.0     0.0   
1    49.0  1.0     140.0  187.0  0.0    172.0    0.0      0.0     0.0     0.0   
2    35.0  1.0     110.0  257.0  0.0    140.0    0.0      0.0     0.0     1.0   
3    54.0  1.0     108.0  309.0  0.0    156.0    0.0      0.0     0.0     1.0   
4    46.0  1.0     130.0  222.0  0.0    112.0    0.0      0.0     0.0     0.0   
..    ...  ...       ...    ...  ...      ...    ...      ...     ...     ...   
721  65.0  1.0     136.0  248.0  0.0    140.0    1.0      4.0     0.0     0.0   
722  28.0  1.0     130.0  132.0  0.0    185.0    0.0      0.0     0.0     1.0   
723  40.0  1.0     130.0  281.0  0.0    167.0    0.0      0.0     0.0     0.0   
724  53.0  1.0     123.0  282.0  0.0     95.0    1.0      2.0     0.0     0.0   
725  59.0  1.0     125.0  238.0  0.0    175.0    0.0      2.6     0.0     0.0   

     ...  slope_1.0  slope_

In [8]:
X_train_lr[continuous_cols] = scaler.fit_transform(X_train_lr[continuous_cols])
X_test_lr[continuous_cols] = scaler.transform(X_test_lr[continuous_cols])

print(X_train_lr)
print(X_test_lr)

          age  sex  trestbps      chol  fbs   thalach  exang   oldpeak  \
0    2.175935  1.0  0.422489 -0.149934  1.0 -1.696222    0.0 -0.876122   
1   -0.497222  1.0  0.422489 -1.119920  0.0  1.405535    0.0 -0.876122   
2   -1.994190  1.0 -1.209039  0.238061  0.0  0.133019    0.0 -0.876122   
3    0.037409  1.0 -1.317808  1.246846  0.0  0.769277    0.0 -0.876122   
4   -0.818001  1.0 -0.121353 -0.440929  0.0 -0.980432    0.0 -0.876122   
..        ...  ...       ...       ...  ...       ...    ...       ...   
721  1.213598  1.0  0.204952  0.063463  0.0  0.133019    1.0  2.900985   
722 -2.742673  1.0 -0.121353 -2.186904  0.0  1.922494    0.0 -0.876122   
723 -1.459558  1.0 -0.121353  0.703654  0.0  1.206704    0.0 -0.876122   
724 -0.069517  1.0 -0.502043  0.723054  0.0 -1.656456    1.0  1.012431   
725  0.572041  1.0 -0.393275 -0.130534  0.0  1.524833    0.0  1.578997   

     cp_1.0  cp_2.0  ...  slope_1.0  slope_2.0  slope_3.0  thal_3.0  thal_6.0  \
0       0.0     0.0  ...      

In [9]:
X_train.columns.to_list()
X_test.columns.to_list()

['age',
 'sex',
 'cp',
 'trestbps',
 'chol',
 'fbs',
 'restecg',
 'thalach',
 'exang',
 'oldpeak',
 'slope',
 'ca',
 'thal']

In [10]:
y_train.to_csv(os.path.join(SPLITS_DIR, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(SPLITS_DIR, 'y_test.csv'), index=False)
X_train_lr.to_csv(os.path.join(SPLITS_DIR, 'X_train_lr.csv'), index=False)
X_test_lr.to_csv(os.path.join(SPLITS_DIR, 'X_test_lr.csv'), index=False)
X_train.to_csv(os.path.join(SPLITS_DIR, 'X_train_rf.csv'), index=False)
X_test.to_csv(os.path.join(SPLITS_DIR, 'X_test_rf.csv'), index=False)